## Synth-EHR demo
    As we mentioned earlier, evaluating accuracy of AI workflows for open-ended task, like summarizing a patient's history relative to a specific appointment or condition, can often be time intensive and require a human in the loop. This is largely because individual patients records can become so complicated that the exact set of tasks an auditer needs to perform to evaluate a task outcome may vary greatly between patients, even if the task and database layout remain the same.

    The goal of this project is to allow a user to test AI agents in a secure context that accurately captures the complexity of navigating a one-size-fits-all EHR when patient histories are complex and varied, but also allow for large-scale evaluation of AI in that context without necessarily needing a human to hand-audit each open-ended task. 

    We allow for data complexity while maintaining a scalable evaluation framework by building our patients, and then building our electronic health record. 
    - We generate patients as individual, referencable dictionaries. Where the conditions they experience at a given time point, and the progression of events that led to those conditions, are all stored together. 
    - We allow a user to describe their own database layout. What tables will exist, and what information will go in them.
    - We populate the database with the patients we generate. 
    Following these steps gives us a testing ground where an AI agent can complete tasks without knowing the correct answer (the database) and a datastructure that can be algorithmically searched for the correct answer regardless of the exact database structure. 

    This notebook outlines our exact workflow. It demonstrates how we can model a complex electronic health record while maintaining an objective ground truth. We will.
    - Generate patient dictionaries
    - Build multiple databases
    - Provide tools for accessing these databases
    - Use our patient dictionaries to evaluate how effectively AI agents perform tasks involving our databases.
    
## Using this notebook

    Currently, running this notebook fully requires configuring a .env with agent API-keys and hosting a FastAPI instance and SQLite database on your computer. You can follow the steps in the readme to set that up, but I recommend simply reading this notebook as is. No need to run any code.

## Step 0: Load required libraries for running agent workflows
Because we are using python to call our agents and run their workflows, we need to import certain software libraries. We do that below, feel free to completely ignore if you are just trying to understand the overall process.

In [1]:

import json
from pathlib import Path
import os
import traceback
import sqlite3
import sys
from pathlib import Path

#run scripts from root, not mvp
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "mvp":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))


## Step 1: Generate patients
As mentioned in the readme.md, the first step is to generate patients. Doing so requires downloading the synthea java application. 
So for this demonstration, we are going to use a pre-generated patient, Carter MacGyver. You can take a closer look at his record at mvp/demo_patients/json/b7e15ab8-7633-df03-b152-d68acebedb56/patient.json to get his entire medical history. The organization structure for Carter's dictionary is shown below. This dictionary contains Carter's entire medical history in 1 single place. Its structure enables rapid lookup of any data.

In [ ]:
'''
Abridged version of Carter's record
'''
{
  "patient": {
    "id": "pat_b7e15ab8-7633-df03-b152-d68acebedb56",
    "firstName": "Carter549",
    "lastName": "MacGyver246",
    "dob": "YYYY-MM-DD",
    "gender": "M",
    "entities": [],
    "birthdate": "1955-02-24",
    "deathdate": "1992-06-19"
},
"allergies": [
    ...
],
"careplans": [
    ...
],
"devices": [
    ...
],
"encounters":[
    ...
],
"imaging_studies":[
    ...
],
"conditions": [
    ...
],
"immunizations":[
    ...
],
"medications":[
    ...
],
"procedures":[
    ...
],
"observations":[
    ...
]
}

## Step 2: Create databases for housing patient information
Now that we have our "ground truth" for the conditions our patient is experiencing, the progression that led to them, and all other medical data, we need to distribute this data across a database. We follow the typical process for organizing an electronic health record. We specify a database layout, and then fit our existing patients to it as well as we can. 

In this example, we specify multiple database layouts, noramlized_v1, and flat_v1. They contain the same information, but organized differently. We do this to show that we can create functional tools and determine optimal workflows regardless of the exact database layout specified by the user.

In [ ]:
from rebuild_db import rebuild_db
CARTER_MACGYVER_FILE = Path("demonstration_patient/demo_patients/json/b7e15ab8-7633-df03-b152-d68acebedb56/patient.json")
SCHEMA_NAMES = ["normalized_v1", "flat_v1"]
DATABASE_DIRECTORY = "mvp/guided_example/demonstration_databases"
SCHEMA_PATH_PREFIX = "mvp/guided_example/demo_schemas"

#Our rebuild_db function builds a database using a layout specified in "" and returns a link to that database
normalized_db_path = rebuild_db("normalized_v1")
flat_db_path = rebuild_db("flat_v1")


normalized_v1 database rebuilt successfully.
flat_v1 database rebuilt successfully.


## Step 3: Distribute patient data across databases
The previous step created the databases themselves, now we insert Carter MacGyver's information into each database.
- We load the adapters for each database. Adapters map goal-directed requests (find a patient, insert a patient) to the exact layout of a specific database. They are generated based on the layout of a database.
- We open the dictionary we made for Carter
- We connect to each database, and use our adapters to distribute Carter's information across the tables that make up each database.

In [3]:
from mvp import load_patients_sqlite3
from mvp.rebuild_db import rebuild_db
from mvp.schema_adapters import flat_v1, normalized_v1

with open(CARTER_MACGYVER_FILE, "r") as file: #load Carter's information
                carter = json.load(file)

conn = sqlite3.connect(normalized_db_path) #connect to DB
normalized_v1.insert_patient(conn, carter) #distribute Carter's info according to this database layout

conn = sqlite3.connect(flat_db_path)
flat_v1.insert_patient(conn, carter)

conn.close() #close database connections

## Step 4: Outline the tools for our AI agent
This step is where we recreate the typical back and forth between an AI agent and an EHR. Just as a typical EHR application provides buttons and links for a provider to access specific patient data, we provide our agent tools that return data without giving the agent full access to our database. 

The tools outlined below don't represent all the tools we provide for the agent in this project, they are just examples of a basic tools that retrieves a single patient from out database. We show a tool formatted for a Claude agent and a tool formatted for an Open AI agent. 

While there are some small formatting differences in how we describe tools for different AI agents. Note that these 2 tools have a lot in common.
- Both provide a description of the tool's purpose and output
- Both provide the input information needed to use a tool
- Neither describe what exactly is happening in our database to retrieve the tool's output
- Neither require any information about how our database is laid out, they function regardless of whether we are using a flat or normalized database



In [ ]:
'''
Anthtropic tools
'''
anthropic_get_patient = {
        "name": "get_patient",
        "description": "Retrieve demographic information for a patient.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "A patient's unique identifier."
                }
            },
            "required": ["patient_id"]
        }
        
    }

'''
OpenAI tools
'''
openai_get_patient = {
        "type": "function",
        "name": "get_patient",
        "description": "Retrieve demographic information for a patient.",
        "parameters": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "A patient's unique identifier."
                }
            },
            "required": ["patient_id"]
        }
        
    }

## Step 5: Outline the task for our agent
Here we outline a framework for specifying a task and evaluation metrics for that task. Each task is defined by an initial prompt (prompt), a list of tools that the agent is provided (allowed_endpoints), the ideal sequence of tool usages that represent a compliant workflow (ideal_workflow) an algorithm that uses our patient dictionaries to find the true correct answer (ground_truth_source), and alogrithms we will use for grading our agent's performance (results_evaluation) and compliance (workflow_evaluation).

Notice that we define our workflow in terms of our agent tools, not by what exact tables and patient records were accessed. That means this evaluation is tied to our task, not to our specific database layout.

Read the prompt below for the details of our demo task.

In [ ]:
medication_retrieval_task = {
    "HELP": {
    "prompt": "LIST: structured messages for the AI agent, each message is a dict with keys 'role' and 'content'",
    "ground_truth_source": "STR: the function whose results will be compared to AI agent's for performance evaluation",
    "allowed_endpoints": "LIST: api endpoints that are accessible to the agent",
    "ideal_workflow": "LIST: ordered sequence of API calls that correctly accomplishes the tast, any deviations will reduce performance score",
    "workflow_evaluation": "STR: the method for assessing whether the ideal workflow was followed",
    "results_evalutation": "STR: the method for assessing results accuracy"
},
    "medication_retrieval_v1": {
    "prompt": [
        {
            "role": "system",
            "content": "You are an AI assistant with access to a synthetic Electronic Health Record API. Use the provided functions whenever you need patient information." 
        },
        {
            "role": "user",
            "content": "List the codes of all medications that the patient with the following ID is currently taking, separate each code with a newline character: "
        },
        {
            "role": "user",
            "content": "Use the available patient and medication information to determine whether a medication should be considered current."
        },
        {
            "role": "user",
            "content": "If you find any current meds, do not return any text besides the list of codes. Otherwise, if a patient has no current meds, return 'No current meds found.' and then a short explanation of how you came to that conclusion"
        }

    ],
    "ground_truth_source": "get_patient_meds(patient_id)",
    "allowed_endpoints": [
      "get_patient",
      "get_medications"
    ],
    "ideal_workflow": [
        "get_patient", "get_medications"
    ],
    "workflow_evaluation": "sequence_match",
    "results_evaluation": "set_match"
  }
}

## Step 6: Run our agents
Our databases have been built, we have given our AI agents tools to access data, and we have described the task we want our agent to complete. The process of assembling these tools and tasking our AI agents is captured in run_agent. Run_agent will return the results of AI agent's task. To demonstrate that we can use the same task regardless of database layout, we run it in our normalized and flat databases. To demonstrate we can use the same task regardless of our agent, we run it for both Claude and OpenAI agents. 

Below we show the layout of an agent's results dictionary and the process of running a task for each agent, along with some placeholder values.

### Note
If you want a more detailed look on how these agents carry out tasks without knowing our database layout, please take a look at run_agent.py.

In [5]:
AGENT_REPORT_SCHEMA = {
        "run_id": "some series of numbers",
        "task": "name of task file",
        "schema": "database layout",
        "agent": "claude/open_ai",
        "provider" : "Anthropic/Open AI",
        "model": "Claude Sonnet-5/gpt-5",
        "patient_id": "carter's patient_id",
        "datetime": "current date and time",
        "tools_workflow": "the list of tools calls made by the agent, in order",
        "workflow_metrics": "grade assigned to the agent's workflow",
        "api_calls_made": "total number of tool calls by agent",
        "total_tokens_used": "total number of tokens used to complete task",
        "total_rows_retrieved": "total number of rows retrieved by agent, each row represents a patient/medication record",
        "raw_response": "unformatted output of agent task",
        "patient_gt": "data that should be returned by this task if done correctly", 
        "output_metrics": "grade assigned to accuracy of agent's data"
    }

from mvp.agents.agent_common import run_agent

claude_analytics_normalized = run_agent(task = "medication_retrieval_v1", curr_agent = "claude", patient_id = carter["patient"]["id"], schema = "normalized_v1")
claude_analytics_flat = run_agent(task = "medication_retrieval_v1", curr_agent = "claude", patient_id = carter["patient"]["id"], schema = "flat_V1")

openai_analytics_normalized = run_agent(task = "medication_retrieval_v1", curr_agent = "open_ai", patient_id = carter["patient"]["id"], schema = "normalized_v1")
openai_analytics_flat = run_agent(task = "medication_retrieval_v1", curr_agent = "open_ai", patient_id = carter["patient"]["id"], schema = "normalized_v1")

SCHEMA HAS BEEN SET TO normalized_v1
<class 'dict'>
{'prompt': [{'role': 'system', 'content': 'You are an AI assistant with access to a synthetic Electronic Health Record API. Use the provided functions whenever you need patient information.'}, {'role': 'user', 'content': 'List the codes of all medications that the patient with the following ID is currently taking, separate each code with a newline character: '}, {'role': 'user', 'content': 'Use the available patient and medication information to determine whether a medication should be considered current.'}, {'role': 'user', 'content': "If you find any current meds, do not return any text besides the list of codes. Otherwise, if a patient has no current meds, return 'No current meds found.' and then a short explanation of how you came to that conclusion"}], 'ground_truth_source': 'get_patient_meds(patient_id)', 'allowed_endpoints': ['get_patient', 'get_medications'], 'ideal_workflow': ['get_patient', 'get_medications'], 'workflow_eval

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /patients/pat_b7e15ab8-7633-df03-b152-d68acebedb56/medications?schema=normalized_v1 (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))

## Special Note
You may be wondering why the database schema is being passed to the agent if the agent is supposed to be unaware of the database's layout. In this context, the schema being passed is simply a label telling the agent which adapter to use, it contains no information about the database layout or data therein. 

## Step 7: Measure compliance
This application defines compliance as the degree to which the agent followed the ideal workflow. The ideal workflow is the sequence of tool calls that fully addresses the task in question without accessing extranneous data. This ideal workflow is defined by the user in the "ideal_workflow" section of our task.

In this demo, we define the ideal workflow as get_patient followed by get_medications. In essence, the ideal workflow involves accessing the patient's basic information such as gender and birthdate, and then that patient's medications.

Similar to run_agent, the logic for evaluation is contained in analyze_workflow. We give the results of our agent's task, and analyze_workflow tells us if any tools were used that shouldn't have been used, if any tools that needed to be used were ommitted, and if the ordering of tools used by the agent matches our ideal workflow.

### Note
We score workflow compliance separately from results accuracy. Naturally, it is possible for the agent to return the correct data, but access data that is not necessary for this workflow.

In [6]:
from mvp.calc_workflow_metrics import analyze_workflow

claude_analytics_normalized["workflow_metrics"] = analyze_workflow(claude_analytics_normalized)
claude_analytics_flat["workflow_metrics"] = analyze_workflow(claude_analytics_flat)

openai_analytics_normalized["workflow_metrics"] = analyze_workflow(openai_analytics_normalized)
openai_analytics_flat["workflow_metrics"] = analyze_workflow(openai_analytics_flat)

NameError: name 'claude_analytics_normalized' is not defined

## Step 8: Determine correct results
Now we leverage the data dictionary we made for our patient to determine the "ground truth" correct data that should be returned from this task. Whereas the agent used the tools we gave it to navigate a database to attempt to find the correct answer, we use our knowledge of the patient's dictionary to quickly arrive at the correct answer. We utilize the following assumption: while database layouts will vary, our patient dictionaries follow a consistent organization structure.

### Note
We only need to run the get_meds function once. No matter what agent we use, or what database layout we use, the correct answer to "what medicines are this patient currently taking" remains the same.

In [ ]:
from .get_patient_meds import get_meds

current_patient_meds = get_meds(carter)

##We update the results of our agent tasks with the true correct answer for easy comparison
claude_analytics_normalized["patient_gt"] = current_patient_meds
claude_analytics_flat["patient_gt"] = current_patient_meds

openai_analytics_normalized["patient_gt"] = current_patient_meds
openai_analytics_flat["patient_gt"] = current_patient_meds

## What meds is the patient currently taking?
Our get_meds function checked Carter's record to see that our simulated patient actually passed away in 1990, and thus is not taking any medications currently. Looking at the record, one would see some medications that have a prescribed date, but no end date. One may incorrectly conclude that these medications are being taken currently, unless they verified that the patient is deceased.

## Step 9: Calculate accuracy of results returned by our agent
For this step, we compare the list of medications returned by our agent to the list we created with get_meds.

Specifically, we measure the following
- Precision: ratio of correct medications returned by agent to total number of medications returned by agent, penalizes returning meds that are not actually current
- Recall: ratio of correct medications returned by agent to total number of medications patient is currently taking, penalizes ommitting meds that are actually current
- F1: overall performance metric that considers precision and recall
- hallucinations: medications that were returned but are not actually current
- missed items: medications that are current but were missed by agent

Similar to before, the logic is contained in the calc_metrics function, which itself contains many lines of code to capture our metrics. If you want a more detailed look at this process, check out calc_output_metrics.py

In [ ]:
from .calc_output_metrics import calc_metrics

claude_analytics_normalized["output_metrics"] = calc_metrics(current_patient_meds, claude_analytics_normalized["raw_response"])
claude_analytics_flat["output_metrics"] = calc_metrics(current_patient_meds, claude_analytics_flat["raw_response"])

openai_analytics_normalized["output_metrics"] = calc_metrics(current_patient_meds, openai_analytics_normalized["raw_response"])
openai_analytics_flat["output_metrics"] = calc_metrics(current_patient_meds, openai_analytics_flat["raw_response"])

## Step 10: Finally, take a look at our agent's performance
We haven now allowed our agent's to attempt the assigned task and graded them. We can look at these results below.

These results highlight a few interesting findings. 

### Workflow steps varied between agents given the same prompts and task
We see that the Claude agent did not follow the ideal workflow outlined in the task. It neglected to call get_patient in either iteration of the task.
We see that the Open AI agent did follow the ideal workflow. Calling get_patient and get_medications, but failed to follow the ideal sequence in the normalized database
It is worth noting that the prompt we passed our agents did not give any indication about a correct workflow, for this illustration we intentionally left it vague to highlight potential differences this process can find.
### Despite taking different approaches, both agents arrived at the correct conclusion
Both agents correctly found that the patient is not taking any current medications. This highlights the importance of verifying workflow adherence and results accuracy separately. In this case, we saw the Claude arrived at the correct answer despite failing to check the patient's basic information, this result is worth attempting to reproduce to get more insight on how this agent reasons about incomplete records.

### Overall results vary between agents and databases, even when the underlying patient information remains the same
This result suports a common intuition, that AI performance varies based on the layout of an electronic health record. This highlights the importance of testing agents on databases that mimic the organization of actual electronic health record systems. 

In [ ]:
'''
Anthropic results
'''
{
  "run_id": "medication_retrieval_v1-20260714T021044.686215Z-23cc",
  "task": "medication_retrieval_v1",
  "schema": "normalized_v1",
  "agent": "claude",
  "provider": "Anthropic",
  "model": "Claude Sonnet-5",
  "patient_id": "pat_b7e15ab8-7633-df03-b152-d68acebedb56",
  "datetime": "2026-07-14T02:10:44.686215+00:00",
  "tools_workflow": [
    "get_medications"
  ],
  "workflow_metrics": {
    "workflow_sequence_match": false,
    "workflow_precision": 1.0,
    "workflow_recall": 0.5,
    "added_steps": [],
    "ommitted_steps": [
      "get_patient"
    ]
  },
  "api_calls_made": 1,
  "total_tokens_used": 3609,
  "total_rows_retrieved": 9,
  "raw_response": "null",
  "patient_gt": [],
  "output_metrics": {
    "precision": 0.0,
    "recall": 0.0,
    "f1": 0.0,
    "hallucinated_items": {
      "null": 1
    },
    "missed_items": {}
  }
}
{
  "run_id": "medication_retrieval_v1-20260714T021115.072175Z-2bd1",
  "task": "medication_retrieval_v1",
  "schema": "flat_v1",
  "agent": "claude",
  "provider": "Anthropic",
  "model": "Claude Sonnet-5",
  "patient_id": "pat_b7e15ab8-7633-df03-b152-d68acebedb56",
  "datetime": "2026-07-14T02:11:15.072175+00:00",
  "tools_workflow": [
    "get_medications"
  ],
  "workflow_metrics": {
    "workflow_sequence_match": false,
    "workflow_precision": 1.0,
    "workflow_recall": 0.5,
    "added_steps": [],
    "ommitted_steps": [
      "get_patient"
    ]
  },
  "api_calls_made": 1,
  "total_tokens_used": 3292,
  "total_rows_retrieved": 9,
  "raw_response": "No current meds found.\n\nExplanation: All medications in this patient's record have either an endDate in the past matching the startDate (single-day treatments, e.g., dental fluoride gel applications) or an explicit endDate that falls before the present date (e.g., the zidovudine courses ending in 1991). The only medication without an endDate is didanosine (started 1991-11-26); however, examining the record as a whole, this entry appears to be the final documented medication in a decades-old history (last activity in 1991), with no subsequent encounters or renewals indicating ongoing treatment. Given the patient's overall record ends in the early 1990s and there is no evidence of continued dispensing, follow-up, or a current encounter supporting active use, none of the medications qualify as currently active.",
  "patient_gt": [],
  "output_metrics": {
    "precision": 1,
    "recall": 1,
    "f1": 1,
    "hallucinated_items": {},
    "missed_items": {}
  }
}

'''
OpenAI results
'''
{
  "run_id": "medication_retrieval_v1-20260714T021050.253384Z-b0dd",
  "task": "medication_retrieval_v1",
  "schema": "normalized_v1",
  "agent": "open_ai",
  "provider": "Open AI",
  "model": "gpt-5",
  "patient_id": "pat_b7e15ab8-7633-df03-b152-d68acebedb56",
  "datetime": "2026-07-14T02:10:50.253384+00:00",
  "tools_workflow": [
    "get_medications",
    "get_patient"
  ],
  "workflow_metrics": {
    "workflow_sequence_match": false,
    "workflow_precision": 1.0,
    "workflow_recall": 1.0,
    "added_steps": [],
    "ommitted_steps": []
  },
  "api_calls_made": 2,
  "total_tokens_used": 6986,
  "total_rows_retrieved": 15,
  "raw_response": "No current meds found.\nAll recorded medications have end dates in the past or, for the one without an end date, the patient died on 1992-06-19, so none are current.",
  "patient_gt": [],
  "output_metrics": {
    "precision": 1,
    "recall": 1,
    "f1": 1,
    "hallucinated_items": {},
    "missed_items": {}
  }
}
{
  "run_id": "medication_retrieval_v1-20260714T021133.556736Z-783e",
  "task": "medication_retrieval_v1",
  "schema": "flat_v1",
  "agent": "open_ai",
  "provider": "Open AI",
  "model": "gpt-5",
  "patient_id": "pat_b7e15ab8-7633-df03-b152-d68acebedb56",
  "datetime": "2026-07-14T02:11:33.556736+00:00",
  "tools_workflow": [
    "get_patient",
    "get_medications"
  ],
  "workflow_metrics": {
    "workflow_sequence_match": true,
    "workflow_precision": 1.0,
    "workflow_recall": 1.0,
    "added_steps": [],
    "ommitted_steps": []
  },
  "api_calls_made": 2,
  "total_tokens_used": 3072,
  "total_rows_retrieved": 15,
  "raw_response": "No current meds found.\nThe patient died on 1992-06-19; all medications either have end dates before that date or, in the case with no end date, would have ended at death, so none are current.",
  "patient_gt": [],
  "output_metrics": {
    "precision": 1,
    "recall": 1,
    "f1": 1,
    "hallucinated_items": {},
    "missed_items": {}
  }
}